# POLITE — 2026-09-07 observation

POLITE's first pointed night. The whole program is four reviewed
YAML plans; this notebook launches them, watches frames land, and inspects what
arrived. It never opens the camera itself.

**Goal:** a usable V-band flat field and a four-star polarimetric calibration, plus
the 50-bias set that closes Mode 5 / gain 56 / offset 20 read noise in electrons.
**Not tonight:** the even-illumination panel (not installed), science targets,
Mode 3 read noise (deferred — it needs its own bias set and its own PTC light
source), reduction-pipeline tweaks.

Operator sheet with the sky table, timeline, and target windows:
`night_plans/20260907_night_sheet.md`.

## Operating rules

- **One acquisition path.** `scripts/execute_night.py <plan> --run` owns the camera,
  EFW, HWP, and mount for the duration of a block. Do not run a capture cell from
  `notebooks/templates/capture.ipynb` while a plan is running.
- **Every `live.*` cell is read-only** — it reads FITS off disk and touches no
  hardware, so it is safe to run while a plan is in flight.
- **Cells that command motion are commented out.** Uncomment deliberately, one at a
  time, after reading the cell above it.
- **`FITSDATA/` is never modified.** Each invocation writes into
  `FITSDATA/20260907/<subdir>/`. Give a repeated block a *new* subdir.
- **PWI4 `alt` is altitude** (0 = zenith, 90 = horizon). The shed-safe window
  is 3–42°, checked after the slew by the runner; a violation aborts that invocation.

## TONIGHT AS ACTUALLY RUN — revised 20:00 PDT

**Twilight flats did not happen.** Nautical twilight passed at 19:57 while the Alpaca
servers were down and the runner's HWP connect was broken. §4 below does **not** run.

**Consequence for reduction:** no flat means the `lsq` default is compromised. Reduce
with **`double_ratio`**, the documented bad-flat-night fallback — dual-beam double ratio
cancels flat-field response and transparency to first order. The standards are still good.

**Order tonight — clear sky is the perishable resource, darks are not:**

| When | What | Why |
|---|---|---|
| 20:00 | **standards1** — run now | ~1 h of clear sky; 9–11 min per target |
| when sky closes | **darkcal** — and it parks | cloud- and time-immune, do it last |

Twenty minutes gets the whole first-order calibration: HD 154345 (unpolarized → zero
point) then HD 183143 (polarized → PA zero + modulation efficiency). Targets after that
are confirmation. Plan order already front-loads them, so it runs unmodified.

```
python scripts/execute_night.py night_plans/20260907_standards.yaml --run \
  --subdir standards1 --yes --no-mount-home

python scripts/execute_night.py night_plans/20260907_darkcal.yaml --run \
  --mount on --park-on-finish
```

An aborted run never parks. If you Ctrl-C the standards, still run darkcal (or park by hand).

**Convention settled 2026-09-07:** PWI4 altitude is conventional — **90 = zenith**, and the
shed-safe window is **altitude 42–90°**. Every earlier 'zenith distance 3–42' statement is dead.


## 0 · Shared preamble

Run these two cells first. They are byte-identical to the template family.

In [ ]:
import os, sys
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / 'pyproject.toml').exists() and _root.parent != _root: _root = _root.parent
if not (_root / 'pyproject.toml').exists(): raise RuntimeError('Could not locate the POLITE repository root')
os.chdir(_root)
if str(_root) not in sys.path: sys.path.insert(0, str(_root))
print('POLITE root:', _root)


In [ ]:
from pathlib import Path
from obs_utils import live
from obs_utils import interactive as obs
from obs_utils import user_config as uc
SESSION_DIR = None  # set in Tonight's card


## 1 · Tonight's card

Provenance, not a camera override: the detector operating point lives in each plan's
`camera:` block (Mode 5 / gain 56 / offset 20, cooler −10 °C) and the runner reads it
back off the hardware before the first frame.

In [ ]:
import subprocess, shutil

NIGHT = '20260907'
PROGRAM = 'standards + twilight flats + Mode 5 read noise'
SESSION_DIR = Path('FITSDATA') / NIGHT
SESSION_DIR.mkdir(parents=True, exist_ok=True)

PYTHON = Path('/Users/blu3/miniforge3/envs/POLITE/bin/python')
if not PYTHON.exists(): PYTHON = Path(sys.executable)

PLANS = Path('night_plans')
FLATS      = PLANS / '20260907_twilight_flats.yaml'
STANDARDS  = PLANS / '20260907_standards.yaml'
STANDARDS2 = PLANS / '20260907_standards_pass2.yaml'
DARKCAL    = PLANS / '20260907_darkcal.yaml'

# subdir -> the flags the night sheet specifies for that invocation
BLOCKS = {
    'twiflat':    (FLATS,      ['--no-mount-home']),
    'standards1': (STANDARDS,  ['--no-mount-home']),
    'standards2': (STANDARDS2, ['--no-mount-home']),
    'darkcal':    (DARKCAL,    ['--mount', 'on', '--no-mount-home', '--park-on-finish']),
}

print('session:', SESSION_DIR, '| program:', PROGRAM)
print('python :', PYTHON)

In [ ]:
def runner_command(plan, *, subdir=None, run=False, extra=()):
    """The exact argv for one execute_night invocation. No --setpoint: every plan
    carries its own `camera:` block and the runner reads it from there."""
    cmd = [str(PYTHON), 'scripts/execute_night.py', str(plan)]
    if run:
        cmd += ['--run', '--yes']
    if subdir is not None:
        cmd += ['--subdir', subdir]
    return cmd + list(extra)


def block_dir(subdir):
    """Where a given --subdir writes. Explicit --subdir means no _HHMM suffix."""
    return SESSION_DIR / subdir


def preview(subdir):
    """Read-only dry-run. Touches no hardware."""
    plan, extra = BLOCKS[subdir]
    subprocess.run(runner_command(plan, subdir=subdir, extra=extra), cwd=Path.cwd(), check=True)


def launch(subdir):
    """MOTION. Start the runner in the BACKGROUND so the live cells below can watch
    frames land. Returns the Popen handle; its stdout is teed to a log file.

    Interrupting the kernel does NOT stop the child -- use proc.terminate(), or run
    the block from a terminal instead (that is the authoritative path for the
    unattended chain)."""
    plan, extra = BLOCKS[subdir]
    cmd = runner_command(plan, subdir=subdir, run=True, extra=extra)
    logs = SESSION_DIR / 'logs'; logs.mkdir(parents=True, exist_ok=True)
    log_path = logs / f'notebook_{subdir}.log'
    print('MOTION -- runner owns camera/EFW/HWP/mount:')
    print(' ', ' '.join(cmd))
    print('  log:', log_path)
    handle = open(log_path, 'w')
    return subprocess.Popen(cmd, cwd=Path.cwd(), stdout=handle, stderr=subprocess.STDOUT, text=True)


def tail(subdir, n=40):
    """Last n lines of a launched block's runner output."""
    log_path = SESSION_DIR / 'logs' / f'notebook_{subdir}.log'
    if not log_path.exists():
        print('no log yet:', log_path); return
    print(''.join(log_path.read_text(errors='replace').splitlines(keepends=True)[-n:]))

In [ ]:
# Pre-flight: the plans exist, and there is room for 811 frames x 52.9 MB ~= 43 GB.
for subdir, (plan, extra) in BLOCKS.items():
    print(f"{subdir:<11s} {plan}  {'OK' if plan.exists() else 'MISSING'}  {' '.join(extra)}")

free_gb = shutil.disk_usage(SESSION_DIR).free / 1024**3
print(f'\nfree on the data volume: {free_gb:.1f} GB  (need >= 50 GB)')
if free_gb < 50:
    print('NOT ENOUGH ROOM -- trim pass 2 (-11 GB) or the 0.3 s rungs (-7 GB), '
          'or free space before 19:05.')

## 2 · Before dark — bring-up and fail-closed gates (18:00, attended)

Connect, verify, cool. Nothing here slews the mount: **home the mount once by hand in
PWI4**, then every plan below runs with `--no-mount-home` so no unattended block
re-homes the mount mid-night.

In [ ]:
# RUN THIS FIRST -- obs.connect_all() only CONNECTS; it starts no server, so with the
# servers down camera, filter wheel and HWP all fail together. Idempotent: each server
# is launched only if its own management endpoint is not already answering.
#   ASCOM Remote :11111 -> ZWO EFW + Optec Pyxis HWP
#   QHY Alpaca   :11112 -> QHY268M camera   (close EZCAP first, USB is exclusive)
# Observatory Windows PC only; skip on the lab Mac.
from obs_utils.alpaca_servers import start_observatory_alpaca_servers
start_observatory_alpaca_servers(
    ascom_endpoint=uc.ALPACA_CONFIG.host,
    qhy_endpoint=uc.ALPACA_CONFIG.camera_host,
)

In [ ]:
s = obs.connect_all()
s.status()

In [ ]:
from obs_utils.night_safety import INSTALLED_EFW_NAMES, verify_filter_wheel
verify_filter_wheel(s.imaging)
print(INSTALLED_EFW_NAMES)

In [ ]:
# The cal blocks are shutterless darks: slot 5 ('Dark') is the only light block the
# QHY268M has. Confirm the driver actually reports it before trusting a BIAS frame.
from obs_utils.night_safety import cooler_gate
cam = s.camera
print('mode/gain/offset now:', cam.ReadoutMode, cam.Gain, cam.Offset)
cam.SetCCDTemperature = -10.0
cooler_gate(cam, -10.0, tol_c=0.5, stable_s=30.0, timeout_s=900.0)

In [ ]:
# MOTION -- home the HWP once per power cycle, on the serial path.
# s = obs.connect_hwp_serial()
# s.home_hwp()

In [ ]:
# MOTION -- prove the stage responds and lands inside tolerance before science.
# from obs_utils.night_safety import HWP_DEFAULT_TOL_DEG
# achieved = s.hwp(22.5)
# print('commanded 22.5, achieved', achieved)
# assert abs(achieved - 22.5) <= HWP_DEFAULT_TOL_DEG
# s.hwp(0.0)

### Mount check — by hand, in PWI4

Enable both axes and home **once, in PWI4 itself**. Every plan below then runs with
`--no-mount-home`, so no unattended block re-homes the mount mid-night. The cell below
is query-only — it reads the current PWI4 altitude and says whether the mount is
inside the shed-safe window right now (being outside is normal when parked). The runner
repeats this check *after* its own slew.

In [ ]:
# Query only -- no connect, home, slew, or capture.
from obs_utils.config import default_sky_regions
from obs_utils.obs_math import zenith_distance_to_altitude, airmass_kasten_young
from obs_utils.pwi4_client import PWI4
from obs_utils.user_config import PWI4_CONFIG

pwi4 = PWI4(host=PWI4_CONFIG.host, port=PWI4_CONFIG.port)
status = pwi4.status()
z = float(status.mount.altitude_degs)
region, = default_sky_regions()
print(f'PWI4 altitude : {z:.2f} deg')
print(f'conventional altitude: {zenith_distance_to_altitude(z):.2f} deg')
print(f'airmass              : {airmass_kasten_young(zenith_distance_to_altitude(z))}')
print(f'allowed PWI4 range   : {region.alt_min_deg:.0f}--{region.alt_max_deg:.0f} deg')
print('axes enabled  :', status.mount.axis0.is_enabled, status.mount.axis1.is_enabled)
print('inside window' if region.alt_min_deg <= z <= region.alt_max_deg
      else 'outside window (normal when parked; the runner checks after its slew)')

## 3 · Dry-run every plan (18:30)

Read-only. Confirm frame counts, HWP angles, targets, the −10 °C setpoint read from
each plan's `camera:` block, and the predicted mount actions. Expected totals:
**twiflat 224 · standards1 291 · standards2 216 · darkcal 80 = 811 frames**.

In [ ]:
for subdir in BLOCKS:
    print('=' * 72); print(subdir); print('=' * 72)
    preview(subdir)

## 4 · Twilight flats — NOT RUN TONIGHT (twilight closed 19:57)

Kept for the record and for the next night. Do not execute these cells tonight.


In [ ]:
preview('twiflat')

In [ ]:
# MOTION -- twilight flats, background launch so the live cells below can watch.
# flats_proc = launch('twiflat')

In [ ]:
tail('twiflat', 30)

In [ ]:
# Read-only. Blocks until the block finishes or timeout_s elapses; interrupt the
# kernel to stop early and keep what was collected. One figure per HWP rung of 4.
flat_stats = live.watch(block_dir('twiflat'), timeout_s=3000, every=4)

### Did the ladder bracket the sky?

`saturated_px` must be 0 and the median wants to sit in the few-thousand-to-~40 000 ADU
range. Rungs that clip are void for flat-fielding; rungs down at the ~50–100 ADU
pedestal carry no signal. The **PTC that converts tonight's read noise to electrons
comes from these frames** — the ≤3 s rungs specifically, because an 8 or 20 s pair
spans ~45 s of a fading sky and that level change inflates the difference variance
(**CONJECTURED**, check it in reduction).

In [ ]:
# Measure once, reuse everywhere. `select` is header-only and returns PATHS; the
# group/trend/histogram calls below accept already-measured FrameStats, so this is the
# only cell that reads 224 full frames off disk.
flat_paths = live.select(block_dir('twiflat'), imagetyp='FLAT')
flats = live.stats_table(flat_paths, show=False)
print(len(flats), 'flat frames measured')
live.group_table(flats, by=('exptime', 'hwp_angle_deg'))

In [ ]:
# Sky fade across the whole ladder, and the exposure/level relation within it.
live.trend(flats, x='time', y='median', by='exptime')
live.trend(flats, x='exptime', y='median')

In [ ]:
live.histogram([st for st in flats if st.exptime == 1.2])

In [ ]:
live.contact_sheet(flats)

In [ ]:
# Explicit clipping roll-up: any non-zero saturated_px voids that rung as a flat.
for st in flats:
    if st.saturated_px:
        print('SATURATED', st.line())
usable = sorted({st.exptime for st in flats if not st.saturated_px and st.median > 1000})
print('rungs with usable levels:', usable)
print('rungs for the PTC (<=3 s, clean pairs):', [e for e in usable if e <= 3])

## 5 · Pointing check — 20:15, attended

The manual gap between the flats and the unattended chain. **Focus is not touched
tonight** — the train is already focused for stellar polarimetry, and a blind sweep
would only risk a known-good position. The one thing this section proves is that the
mount puts the commanded star in the field.

HD 154345 does not enter the shed-safe window until **20:13** (z 42° → 24.9° by 20:30),
so do not slew before then; the altitude gate would refuse the capture anyway.
Confirm both beams of the Savart pair are visible and comparably sharp, then launch §6.
**Do not launch §6 blind** — the unattended chain has no one to notice a mount that
slewed somewhere unexpected.

Plate solving is *not* on the pre-capture path. It moved to §8, after the data is in.

In [ ]:
# MOTION -- slew to the first standard and confirm the field.
# from obs_utils.mount import slew_radec_j2000
# s = obs.connect_mount()
# slew_radec_j2000(s.pwi4, 17.0434455, 47.081879)
# z = float(s.pwi4.status().mount.altitude_degs)
# print(f'post-slew PWI4 altitude: {z:.2f} deg (must be 42--90)')

In [ ]:
# MOTION -- one supervised probe frame, written outside any plan's subdir.
# probe = s.expose(2.0, out_path=SESSION_DIR / 'pointcheck_probe.fits')
# live.frame_report(probe)

## 6 · Standards + calibration — 20:30, unattended

**The authoritative path is a terminal**, so the chain survives a closed laptop or a
dead kernel. Three invocations joined with `;` — never `&&` — so an abort in one does
not cancel the rest:

```zsh
scripts/execute_night.py night_plans/20260907_standards.yaml \
  --run --subdir standards1 --yes --no-mount-home ; \
scripts/execute_night.py night_plans/20260907_standards_pass2.yaml \
  --run --subdir standards2 --yes --no-mount-home ; \
scripts/execute_night.py night_plans/20260907_darkcal.yaml \
  --run --subdir darkcal --yes --mount on --no-mount-home --park-on-finish
```

**Parking rides on `darkcal`, on purpose.** `--park-on-finish` fires only after a clean
finish (`scripts/execute_night.py:441` sits outside the run block, and an abort
propagates past it). `darkcal` commands no slews, so it is the one invocation that
cannot abort on the altitude gate — which makes it the reliable parker even if
both science plans die. Do not reorder the chain.

HD 154345 is first because its window closes at 22:12; it is deliberately absent from
pass 2, where a 22:12+ slew would abort the run. Pass 2 is optional — dropping it costs
nothing but the second-epoch parallactic-angle diagnostic.

In [ ]:
preview('standards1'); preview('standards2'); preview('darkcal')

In [ ]:
# MOTION -- notebook equivalent of the terminal chain. Use the terminal instead unless
# you are staying with the kernel; `;` semantics are reproduced by NOT raising between
# blocks, so an aborted plan does not cancel the ones after it.
# for subdir in ('standards1', 'standards2', 'darkcal'):
#     proc = launch(subdir)
#     rc = proc.wait()
#     print(f'{subdir}: exit {rc}' + ('' if rc == 0 else '  <-- ABORTED, continuing'))

In [ ]:
# Read-only, safe while the chain runs. One figure per complete 8-angle cycle.
science_stats = live.watch(block_dir('standards1'), timeout_s=3600, every=8)

In [ ]:
tail('standards1', 40)

## 7 · Read-only checks after each block

Provenance and completeness only. The instrumental q/u estimate and the
polarized-standard P/PA comparison need tracked paired-aperture photometry and belong
in the dated reduction notebook — do not infer them from whole-frame statistics here.

In [ ]:
for subdir in BLOCKS:
    d = block_dir(subdir)
    n = len(live.find_frames(d))
    print('=' * 72); print(f'{subdir}  ({n} frames)  {d}'); print('=' * 72)
    if n:
        live.session_table(d)

In [ ]:
live.hwp_coverage(block_dir('standards1'))
live.qa_print(live.sequence_audit(block_dir('standards1')))

In [ ]:
# Clipping is the one thing that voids a target's complete HWP cycle. Measure once,
# then group. This reads 291 full frames, so give it a minute.
lights = live.stats_table(live.select(block_dir('standards1'), imagetyp='LIGHT'), show=False)
live.group_table(lights, by=('object_name', 'exptime'))
voided = {(st.object_name, st.exptime) for st in lights if st.saturated_px}
for st in lights:
    if st.saturated_px:
        print('SATURATED --', st.line())
print('\ntarget/exposure cycles voided by clipping:', sorted(voided) or 'none')

In [ ]:
# Modulation should show as a smooth run of median with HWP angle for a polarized
# standard and a flat line for an unpolarized one. Indicative only, NOT a measurement:
# whole-frame medians are not paired-aperture photometry. The real q/u belongs in the
# dated reduction notebook.
live.trend([st for st in lights if st.object_name == 'HD 183143'],
           x='hwp_angle_deg', y='median', by='exptime')

### The Mode 5 read-noise set

`darkcal` carries 50 bias frames at Mode 5 / gain 56 / offset 20. `bias_qa` is the
gate: level and stability, sigma-clipped. The **measurement** — σ of (biasᵢ − biasⱼ)/√2
over many pairs, the Janesick difference-pair method — belongs in the reduction
notebook via `caltools`, not here.

Offset 20 puts the pedestal at roughly 50–100 ADU, so the bias distribution's left tail
is intact and the measured σ is the real read noise. Worth a glance at the histogram;
it is not a gate.

In [ ]:
bias_paths = live.select(block_dir('darkcal'), imagetyp='BIAS')
print(len(bias_paths), 'bias frames (expect 50)')
live.qa_print(live.bias_qa(bias_paths))
live.histogram(bias_paths)   # left tail should clear zero with room to spare

In [ ]:
darks = live.stats_table(live.select(block_dir('darkcal'), imagetyp='DARK'), show=False)
live.group_table(darks, by=('exptime',))
live.trend(darks, x='exptime', y='median')     # slope -> dark rate at Mode 5, -10 C
live.trend(darks, x='time', y='det_temp_c')    # cooler held? drift voids the slope

## 8 · PlateSolve3 — optional, after the data is in

Nothing above depends on this and it must not delay a block or the closeout. It is a
**read-only commissioning probe on frames already written to disk**: record PWI4 state
before and after plus the raw PS3 output fields, and make no mount or pointing-model
change tonight. The solver has never been run on a Savart-doubled field — every star
appears twice — so inspect the gray display before trusting anything it reports.

Skip it entirely if the night ran late, or if the PS3CLI executable and Kepler
catalogue are not installed on this machine. A live solve during acquisition is the
eventual goal and is not tonight's business; tonight only asks whether PS3 returns
anything sane on a POLITE frame.

In [ ]:
# Prefer a real captured standards frame -- the point of solving after the fact is that
# the data is already on disk. Falls back to the attended probe frame if one was taken.
_solve_candidates = (live.find_frames(block_dir('standards1'))
                     or live.find_frames(block_dir('standards2')))
PLATE_FRAME = (Path(_solve_candidates[0]) if _solve_candidates
               else SESSION_DIR / 'pointcheck_probe.fits')
PLATE_SCALE_ARCSEC_PER_PX = 0.224
PS3CLI_EXE = Path(r'C:\SET\PS3CLI\ps3cli.exe')   # set on the observatory PC
PS3_CATALOG = Path(r'C:\SET\Kepler')              # set on the observatory PC
print('frame to solve:', PLATE_FRAME)
if PLATE_FRAME.exists():
    live.show_frame(PLATE_FRAME, cmap='gray', percentile_clip=(5, 99.8))
else:
    print('nothing on disk to solve -- skip this section')

In [ ]:
# Query-only: no slew, no offset, no capture, no model update, no change to the FITS.
# from obs_utils.platesolve import PlateSolveConfig, platesolve
# if s.pwi4 is None: raise RuntimeError('Run obs.connect_all() first; this is query-only.')
# pwi_before = s.pwi4.status()
# plate_result = platesolve(PLATE_FRAME, PLATE_SCALE_ARCSEC_PER_PX, PlateSolveConfig(PS3CLI_EXE, PS3_CATALOG))
# pwi_after = s.pwi4.status()
# print('PWI4 J2000 before/after:',
#       (pwi_before.mount.ra_j2000_hours, pwi_before.mount.dec_j2000_degs),
#       (pwi_after.mount.ra_j2000_hours, pwi_after.mount.dec_j2000_degs))
# print('PS3 raw fields:', dict(plate_result.raw_fields))

## Closeout

The runner closes the camera, EFW, and HWP itself after each block, and `darkcal` parks
the mount. Preserve the raw FITS, `block_manifest.jsonl`, and `pol_config.yaml` from
every subdirectory, plus `FITSDATA/20260907/logs/`.

Record in the observing log: which blocks completed, the usable flat rungs, any
clipping (and which target cycle it voided), whether pass 2 ran, the post-slew PWI4
altitude for each target, whether the PlateSolve3 proof was attempted and what
raw fields it returned, and whether the mount parked.

**Every number in the plan files was a prediction until tonight.** Say plainly in the
log which ones survived contact.

In [ ]:
for subdir in BLOCKS:
    d = block_dir(subdir)
    if live.find_frames(d):
        print('=' * 72); print(subdir); print('=' * 72)
        live.session_table(d)
        live.qa_print(live.sequence_audit(d))

In [ ]:
# Read-only. Whole-night trends across every block that produced frames: per-frame
# detector temperature (reduction uses this, not the setpoint) and level/noise drift.
import matplotlib.pyplot as plt

all_stats = [live.frame_stats(f)
             for subdir in BLOCKS
             for f in live.find_frames(block_dir(subdir))]
print(f'{len(all_stats)} frames across {len(BLOCKS)} blocks')
if all_stats:
    live.temperature_trend(all_stats); plt.show()
    live.level_trend(all_stats); plt.show()

In [ ]:
obs.shutdown()